# 02 — Stem Separation

**Purpose:** Isolate the dialogue track from music and SFX so ASR and TTS operate on clean speech.

## Why this step matters for dubbing
If you feed mixed audio to ASR, the transcription WER roughly doubles on music-heavy scenes. If you feed mixed audio to Gemini ASR it handles it better, but background music still causes hallucinations and timestamp errors. Separating stems first also lets us mix the original music bed back under the dubbed dialogue in notebook 07.

## Model selection rationale

| Model | Architecture | Why included | MUSDB18-HQ SDR |
|---|---|---|---|
| **BS-RoFormer** | Rotary Transformer on full STFT | 2024 SOTA; best objective SDR on vocals | 12.97 dB |
| **BS-RoFormer 1297** (community) | Same arch, community fine-tune | Slightly better than official weights on several test sets | ~13.2 dB est. |
| **MelBand-RoFormer** | Transformer on mel bands | Better at preserving vocal harmonics in upper registers | 11.43 dB |
| **HTDemucs-FT** | Hybrid time+frequency domain CNN | Fastest; good on non-Western music where RoFormer can over-separate | ~9–11 dB |
| **MDX-Net** | ONNX CNN | Fast CPU inference; good fallback if GPU OOM | ~8–10 dB |

## Metrics rationale
- **RTF (Real-Time Factor)** = processing_time / audio_duration. RTF < 1 = faster than real-time. Determines whether this step is a pipeline bottleneck.
- **Downstream WER** is the *most important* metric here — a model with slightly lower SDR but much lower WER on the resulting vocals is the right choice for dubbing.
- **Subjective dialogue clarity & music preservation** (1–5) capture perceptual quality that objective metrics miss.
- **SDR/SIR/SAR** (via mir_eval) require isolated ground-truth stems we don't have; noted as N/A.

## Ensemble strategy
Averaging magnitude spectrograms across BS-RoFormer + MelBand-RoFormer before iSTFT often beats either model alone ("spectrogram averaging" ensemble). Built as a bonus cell at the end.

**Input:** `intermediate/audio_extracted/source_audio.wav`  
**Output:** `intermediate/stems/vocals.wav` + `stems/instrumental.wav` (at source SR)


In [ ]:
import importlib
missing = [p for p in ["audio_separator","librosa","soundfile","pandas"] if not importlib.util.find_spec(p)]
if missing:
    import subprocess, sys
    subprocess.run([sys.executable,"-m","pip","install","-q","audio-separator[cpu]","librosa","soundfile","tqdm","pandas"])
print("Ready.")

In [ ]:
import sys, os, json, time, shutil
os.environ['PATH'] = '/opt/homebrew/bin:' + os.environ.get('PATH', '')
import numpy as np
import librosa
import pandas as pd
from tqdm.notebook import tqdm
from IPython.display import Audio, display

sys.path.insert(0, os.path.abspath('..'))
from config import AUDIO_EXTRACTED_DIR, STEMS_DIR, load_source_sr

SOURCE_WAV    = os.path.join(AUDIO_EXTRACTED_DIR, 'source_audio.wav')
RESULTS_DIR   = os.path.join(STEMS_DIR, 'model_outputs')
os.makedirs(RESULTS_DIR, exist_ok=True)

SOURCE_SR = load_source_sr()
DURATION  = json.load(open(os.path.join(AUDIO_EXTRACTED_DIR, "meta.json")))["duration_seconds"]
print(f'Source: {DURATION:.1f}s at {SOURCE_SR} Hz')

results = {}  # model_name -> {vocals_path, instrumental_path, rtf, error}


In [ ]:
from audio_separator.separator import Separator

def run_separator(model_filename, display_name):
    out_dir   = os.path.join(RESULTS_DIR, display_name.replace(' ', '_'))
    vocals_out = os.path.join(out_dir, 'vocals.wav')
    instr_out  = os.path.join(out_dir, 'instrumental.wav')

    if os.path.exists(vocals_out) and os.path.exists(instr_out):
        print(f'[{display_name}] Cached — skipping')
        results[display_name] = {'vocals': vocals_out, 'instrumental': instr_out, 'rtf': None, 'error': None}
        return

    os.makedirs(out_dir, exist_ok=True)
    print(f'[{display_name}] Starting...')
    t0 = time.time()
    try:
        sep = Separator(output_dir=out_dir, output_format='WAV', normalization_threshold=0.9)
        sep.load_model(model_filename=model_filename)
        sep.separate(SOURCE_WAV)
        elapsed = time.time() - t0

        # ── FIX: robust vocal/instrumental file detection ────────────────────
        # Bug: 'vocal' in fn also matches 'no_vocal'.  We must check 'no_vocal'
        # (and 'instrumental', 'music') FIRST before the bare 'vocal' check.
        found_v = found_i = None
        for fn in os.listdir(out_dir):
            fl = fn.lower()
            if any(k in fl for k in ('instrumental', 'no_vocal', 'no-vocal', 'music', 'accompaniment')):
                found_i = fn
            elif 'vocal' in fl:   # only if not already matched as instrumental
                found_v = fn

        if found_v: shutil.move(os.path.join(out_dir, found_v), vocals_out)
        if found_i: shutil.move(os.path.join(out_dir, found_i), instr_out)

        rtf = elapsed / DURATION
        print(f'  Done in {elapsed:.0f}s  RTF={rtf:.2f}  ({1/rtf:.1f}x realtime)')
        results[display_name] = {'vocals': vocals_out, 'instrumental': instr_out, 'rtf': rtf, 'error': None}
    except Exception as e:
        print(f'  FAILED: {e}')
        results[display_name] = {'vocals': None, 'instrumental': None, 'rtf': None, 'error': str(e)}

print('Separator helper ready.')


## BS-RoFormer (2024 SOTA)

Published by Chen et al. (2024). Uses Rotary Position Embeddings inside a Transformer that operates directly on STFT spectrograms. Achieves the highest published SDR on MUSDB18-HQ. Downloads ~200 MB checkpoint on first run.


In [ ]:
run_separator('model_bs_roformer_ep_317_sdr_12.9755.ckpt', 'BS-RoFormer-official')


In [ ]:
# Community fine-tune of BS-RoFormer with additional training data.
# Marginally better on some test sets; worth a head-to-head compare.
run_separator('model_bs_roformer_ep_1296_sdr_13.2804.ckpt', 'BS-RoFormer-1297')


## MelBand-RoFormer

Same Transformer backbone as BS-RoFormer but operates in mel-frequency space rather than linear STFT. Better at preserving subtle vocal harmonics above 8 kHz. Slightly lower SDR than BS-RoFormer but perceptually comparable on singing; less tested on speech-heavy content.


In [ ]:
run_separator('model_mel_band_roformer_ep_3005_sdr_11.4360.ckpt', 'MelBand-RoFormer')


## HTDemucs-FT (Meta, Hybrid)

Meta's 4th generation Demucs, fine-tuned on proprietary data. Hybrid architecture: simultaneously processes raw waveform (temporal) and STFT spectrogram (frequency). Fastest of the four on GPU. Good generalisation on non-Western music (Bollywood, Carnatic) where RoFormer models — trained mostly on Western pop — can sometimes over-separate percussion that sits in the vocal frequency range.


In [ ]:
run_separator('htdemucs_ft.yaml', 'HTDemucs-FT')


## MDX-Net (ONNX, fastest CPU baseline)

Winner of the MDX 2021 challenge. CNN-based, exported to ONNX — very fast on CPU (< 0.5x RTF). Lower quality ceiling than the RoFormer variants but useful as a speed benchmark and fallback when GPU memory is limited.


In [ ]:
run_separator('UVR-MDX-NET-Inst_3.onnx', 'MDX-Net')


## Commercial — AudioShake API

AudioShake offers state-of-the-art stem separation as a service, trained on a much larger proprietary dataset than any OSS alternative. Free trial: 10 credits (no credit card required). Sign up at [audioshake.ai](https://audioshake.ai).

**Cost:** 1 credit per audio file. Free trial covers one full test run.

Paste your API key when prompted, or press Enter to skip and test later.

In [ ]:
from getpass import getpass
import requests

AUDIOSHAKE_API_KEY = os.getenv('AUDIOSHAKE_API_KEY') or getpass('AudioShake API key (Enter to skip): ')

AUDIOSHAKE_OUT_DIR = os.path.join(RESULTS_DIR, 'AudioShake')
os.makedirs(AUDIOSHAKE_OUT_DIR, exist_ok=True)
vocals_out = os.path.join(AUDIOSHAKE_OUT_DIR, 'vocals.wav')
instr_out  = os.path.join(AUDIOSHAKE_OUT_DIR, 'instrumental.wav')

if not AUDIOSHAKE_API_KEY.strip():
    print('[AudioShake] Skipped (no API key). Set AUDIOSHAKE_API_KEY env var to test later.')
    results['AudioShake'] = {'vocals': None, 'instrumental': None, 'rtf': None, 'error': 'No API key'}

elif os.path.exists(vocals_out) and os.path.exists(instr_out):
    print('[AudioShake] Cached — skipping')
    results['AudioShake'] = {'vocals': vocals_out, 'instrumental': instr_out, 'rtf': None, 'error': None}

else:
    try:
        t0 = time.time()

        # 1. Upload audio
        with open(SOURCE_WAV, 'rb') as f:
            upload_resp = requests.post(
                'https://groovy.audioshake.ai/upload/',
                headers={'Authorization': f'Bearer {AUDIOSHAKE_API_KEY}'},
                files={'file': (os.path.basename(SOURCE_WAV), f, 'audio/wav')},
            )
        upload_resp.raise_for_status()
        asset_id = upload_resp.json()['id']
        print(f'  Uploaded. Asset ID: {asset_id}')

        # 2. Submit separation job
        job_resp = requests.post(
            'https://groovy.audioshake.ai/job/',
            headers={'Authorization': f'Bearer {AUDIOSHAKE_API_KEY}', 'Content-Type': 'application/json'},
            json={
                'metadata': {'format': 'wav'},
                'outputAssets': [{'type': 'vocals'}, {'type': 'accompaniment'}],
                'inputAssetId': asset_id,
            },
        )
        job_resp.raise_for_status()
        job_id = job_resp.json()['job']['id']
        print(f'  Job: {job_id}. Polling...')

        # 3. Poll for completion
        while True:
            time.sleep(10)
            status_resp = requests.get(
                f'https://groovy.audioshake.ai/job/{job_id}',
                headers={'Authorization': f'Bearer {AUDIOSHAKE_API_KEY}'},
            )
            status = status_resp.json()['job']['status']
            print(f'  Status: {status}')
            if status == 'completed': break
            if status == 'error': raise RuntimeError(f'AudioShake error: {status_resp.json()}')

        # 4. Download stems
        for asset in status_resp.json()['job']['outputAssets']:
            dl = requests.get(asset['link'])
            asset_type = asset['type']
            if 'vocal' in asset_type and 'no' not in asset_type:
                with open(vocals_out, 'wb') as f: f.write(dl.content)
            elif asset_type in ('accompaniment', 'instrumental', 'no_vocals'):
                with open(instr_out, 'wb') as f: f.write(dl.content)

        elapsed = time.time() - t0
        rtf = elapsed / DURATION
        results['AudioShake'] = {'vocals': vocals_out, 'instrumental': instr_out, 'rtf': rtf, 'error': None}
        print(f'  Done: {elapsed:.0f}s  RTF={rtf:.2f}')
    except Exception as e:
        print(f'  AudioShake FAILED: {e}')
        results['AudioShake'] = {'vocals': None, 'instrumental': None, 'rtf': None, 'error': str(e)}

## Commercial — ElevenLabs Audio Isolation

Uses your existing ElevenLabs subscription. Sends the full audio to ElevenLabs' proprietary speech enhancer, which removes background noise and music. **Note:** Audio Isolation returns clean speech only — no separate instrumental. Pair with an OSS model's instrumental if you need both tracks.

In [ ]:
from getpass import getpass
EL_API_KEY_STEM = os.getenv('ELEVENLABS_API_KEY') or getpass('ElevenLabs API key (for Audio Isolation, Enter to skip): ')

EL_ISO_DIR = os.path.join(RESULTS_DIR, 'EL_AudioIsolation')
os.makedirs(EL_ISO_DIR, exist_ok=True)
el_vocals_out = os.path.join(EL_ISO_DIR, 'vocals.wav')

if not EL_API_KEY_STEM.strip():
    print('[EL Audio Isolation] Skipped (no API key).')
    results['EL-AudioIsolation'] = {'vocals': None, 'instrumental': None, 'rtf': None, 'error': 'No API key'}

elif os.path.exists(el_vocals_out):
    print('[EL Audio Isolation] Cached — skipping')
    results['EL-AudioIsolation'] = {'vocals': el_vocals_out, 'instrumental': None, 'rtf': None, 'error': None}

else:
    try:
        t0 = time.time()
        with open(SOURCE_WAV, 'rb') as f:
            resp = requests.post(
                'https://api.elevenlabs.io/v1/audio-isolation',
                headers={'xi-api-key': EL_API_KEY_STEM},
                files={'audio': (os.path.basename(SOURCE_WAV), f, 'audio/wav')},
            )
        resp.raise_for_status()
        with open(el_vocals_out, 'wb') as out:
            out.write(resp.content)
        elapsed = time.time() - t0
        rtf = elapsed / DURATION
        results['EL-AudioIsolation'] = {'vocals': el_vocals_out, 'instrumental': None, 'rtf': rtf, 'error': None}
        print(f'  EL Audio Isolation: {elapsed:.0f}s  RTF={rtf:.2f}')
        print('  No instrumental track — pair with OSS model instrumental for assembly.')
    except Exception as e:
        print(f'  EL Audio Isolation FAILED: {e}')
        results['EL-AudioIsolation'] = {'vocals': None, 'instrumental': None, 'rtf': None, 'error': str(e)}

## Optional: DeepFilterNet3 post-processing

After vocal extraction, the speech can still contain residual music bleed and room noise. DeepFilterNet3 (Schroter et al. 2023) is a neural speech enhancer / denoiser that runs at real-time on CPU. Chain it after the winning model's vocals.


In [ ]:
# Run deepfilternet on the vocal track from the winning model.
# pip install deepfilternet (not in requirements.txt by default — install manually)
try:
    from df.enhance import enhance, init_df, load_audio, save_audio
    import torch

    VOCALS_IN  = results.get('BS-RoFormer-official', {}).get('vocals') or results.get(list(results.keys())[0], {}).get('vocals')
    VOCALS_DNF = os.path.join(STEMS_DIR, 'model_outputs', 'vocals_deepfiltered.wav')

    if VOCALS_IN and os.path.exists(VOCALS_IN):
        model_dnf, df_state, _ = init_df()
        audio, _ = load_audio(VOCALS_IN, sr=df_state.sr())
        enhanced  = enhance(model_dnf, df_state, audio)
        save_audio(VOCALS_DNF, enhanced, df_state.sr())
        results['DeepFilterNet3'] = {'vocals': VOCALS_DNF, 'instrumental': None, 'rtf': None, 'error': None}
        print(f'DeepFilterNet3 post-processed -> {VOCALS_DNF}')
    else:
        print('No vocals source found. Run stem separation models first.')
except ImportError:
    print('DeepFilterNet not installed. Skip or: pip install deepfilternet')
except Exception as e:
    print(f'DeepFilterNet FAILED: {e}')


## Metrics & Comparison

### Why these metrics?
- **RTF** directly predicts pipeline throughput
- **Downstream WER** is the ground truth for what matters in dubbing — does ASR work better on this separation?
- **Subjective** ratings capture music bleed, clipping, and artefacts that SDR misses
- **SDR** (if you have reference stems) is the standard academic benchmark but N/A here


In [ ]:
rows = []
for name, r in results.items():
    rows.append({
        'Model':            name,
        'RTF':              round(r['rtf'], 3) if r['rtf'] else None,
        'Speed (x RT)':     round(1/r['rtf'], 1) if r['rtf'] else None,
        'Vocals exists':    os.path.exists(r['vocals']) if r['vocals'] else False,
        'Instr exists':     os.path.exists(r['instrumental']) if r['instrumental'] else False,
        'Error':            r['error'],
    })
df_rtf = pd.DataFrame(rows)
print(df_rtf.to_string(index=False))


In [ ]:
# ── Automated spectral quality metrics ─────────────────────────────────────
# Works after kernel restart — reads from disk, no results dict needed.
import matplotlib.pyplot as plt
import numpy as np
import librosa
import soundfile as sf

_DISK_MODELS = {
    "BS-RoFormer-official": {
        "vocals": os.path.join(STEMS_DIR, "model_outputs", "BS-RoFormer-official", "vocals.wav"),
        "instrumental": os.path.join(STEMS_DIR, "model_outputs", "BS-RoFormer-official", "instrumental.wav"),
        "rtf": 1.942,
    },
    "MelBand-RoFormer": {
        "vocals": os.path.join(STEMS_DIR, "model_outputs", "MelBand-RoFormer", "vocals.wav"),
        "instrumental": os.path.join(STEMS_DIR, "model_outputs", "MelBand-RoFormer", "instrumental.wav"),
        "rtf": 1.452,
    },
    "HTDemucs-FT": {
        "vocals": os.path.join(STEMS_DIR, "model_outputs", "HTDemucs-FT", "vocals.wav"),
        "instrumental": os.path.join(STEMS_DIR, "model_outputs", "HTDemucs-FT", "instrumental.wav"),
        "rtf": 1.581,
    },
    "MDX-Net": {
        "vocals": os.path.join(STEMS_DIR, "model_outputs", "MDX-Net", "vocals.wav"),
        "instrumental": os.path.join(STEMS_DIR, "model_outputs", "MDX-Net", "instrumental.wav"),
        "rtf": 0.286,
    },
}

quality_rows = []
for name, r in _DISK_MODELS.items():
    vpath = r["vocals"]
    ipath = r["instrumental"]
    if not os.path.exists(vpath):
        print(f"  [{name}] vocals not found — skipping")
        continue
    try:
        y_voc, sr_v = librosa.load(vpath, sr=None, mono=True, duration=60)
        S_voc        = np.abs(librosa.stft(y_voc))
        freqs        = librosa.fft_frequencies(sr=sr_v)
        band         = (freqs >= 300) & (freqs <= 3400)
        speech_ratio = float(np.mean(S_voc[band])) / (float(np.mean(S_voc)) + 1e-9)
        flatness     = float(np.mean(librosa.feature.spectral_flatness(y=y_voc)))
        rms_voc      = float(np.sqrt(np.mean(y_voc ** 2)))
        rms_ins      = None
        if os.path.exists(ipath):
            y_ins, _ = librosa.load(ipath, sr=sr_v, mono=True, duration=60)
            rms_ins  = float(np.sqrt(np.mean(y_ins ** 2)))
        quality_rows.append({
            "Model": name, "RTF": r["rtf"],
            "Speech band ratio": round(speech_ratio, 3),
            "Spectral flatness": round(flatness, 5),
            "Vocal RMS": round(rms_voc, 4),
            "Instr RMS": round(rms_ins, 4) if rms_ins else None,
        })
    except Exception as qe:
        print(f"  [{name}] metrics failed: {qe}")

import pandas as pd
df_quality = pd.DataFrame(quality_rows)
print(df_quality.to_string(index=False))

if len(quality_rows) >= 2:
    names = [r["Model"] for r in quality_rows]
    fig, axes = plt.subplots(1, 3, figsize=(16, max(3.5, len(names)*0.6 + 1.5)))
    rtfs  = [r["RTF"] for r in quality_rows]
    axes[0].barh(names, [1/x if x else 0 for x in rtfs], color="steelblue")
    axes[0].set_xlabel("Speed (x realtime)"); axes[0].set_title("Processing Speed")
    sbrs = [r["Speech band ratio"] for r in quality_rows]
    bars = axes[1].barh(names, sbrs, color="darkorange")
    axes[1].set_xlabel("300-3400 Hz energy ratio"); axes[1].set_title("Speech Band Ratio")
    for bar, val in zip(bars, sbrs): axes[1].text(val+0.005, bar.get_y()+bar.get_height()/2, f"{val:.3f}", va="center", fontsize=8)
    flats = [r["Spectral flatness"] for r in quality_rows]
    bars2 = axes[2].barh(names, flats, color="seagreen")
    axes[2].set_xlabel("Spectral flatness (lower = cleaner)"); axes[2].set_title("Vocal Spectral Flatness")
    for bar, val in zip(bars2, flats): axes[2].text(val+1e-6, bar.get_y()+bar.get_height()/2, f"{val:.5f}", va="center", fontsize=8)
    plt.tight_layout()
    chart_path = os.path.join(STEMS_DIR, "stem_comparison.png")
    plt.savefig(chart_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Chart saved -> {chart_path}")


In [ ]:
# ── Downstream WER check (most important metric) ──────────────────────────────
try:
    import whisper, torch, jiwer

    device = ('cuda' if torch.cuda.is_available()
               else 'mps' if torch.backends.mps.is_available()
               else 'cpu')
    fp16 = device in ('cuda', 'mps')
    print(f'Loading Whisper base for downstream WER check on {device}...')
    wer_model = whisper.load_model('base', device=device)

    wer_results = {}
    for name, r in tqdm(results.items(), desc='WER downstream'):
        if not r['vocals'] or not os.path.exists(r['vocals']):
            continue
        res = wer_model.transcribe(r['vocals'], language=SOURCE_LANGUAGE, task='transcribe', fp16=fp16)
        wer_results[name] = res['text'].strip()
        print(f'  [{name}] {len(res["text"])} chars')

    if wer_results:
        ref_name = list(wer_results.keys())[0]
        ref_text = wer_results[ref_name]
        print(f'\nPseudo-reference: {ref_name}')
        for name, text in wer_results.items():
            if name == ref_name:
                continue
            w = jiwer.wer(ref_text, text)
            print(f'  {name}: pseudo-WER vs ref = {w:.3f}')
except Exception as e:
    print(f'WER downstream check skipped: {e}')

In [ ]:
import soundfile as sf
PREVIEW_SECS = 30
for name, r in results.items():
    if r["vocals"] and os.path.exists(r["vocals"]):
        y_v, sr_v = sf.read(r["vocals"], frames=PREVIEW_SECS * SOURCE_SR)
        print(f"=== {name} — Vocals (first {PREVIEW_SECS}s) ===")
        display(Audio(y_v.T if y_v.ndim > 1 else y_v, rate=sr_v))
        if r["instrumental"] and os.path.exists(r["instrumental"]):
            y_i, sr_i = sf.read(r["instrumental"], frames=PREVIEW_SECS * SOURCE_SR)
            print(f"--- {name} — Instrumental (first {PREVIEW_SECS}s) ---")
            display(Audio(y_i.T if y_i.ndim > 1 else y_i, rate=sr_i))
    else:
        print(f"[{name}] No output available (error or skipped)")

In [ ]:
# ── Subjective ratings (fill after listening) ─────────────────────────────────
# 1 = bad, 5 = excellent.  Fill in after running the listen cell above.
# dialogue_clarity  : how clean is the speech? any music bleed?
# music_preservation: is the instrumental missing instruments or damaged?
subjective = {
    'BS-RoFormer-official': {'dialogue_clarity': None, 'music_preservation': None},
    'BS-RoFormer-1297':     {'dialogue_clarity': None, 'music_preservation': None},
    'MelBand-RoFormer':     {'dialogue_clarity': None, 'music_preservation': None},
    'HTDemucs-FT':          {'dialogue_clarity': None, 'music_preservation': None},
    'MDX-Net':              {'dialogue_clarity': None, 'music_preservation': None},
    'DeepFilterNet3':       {'dialogue_clarity': None, 'music_preservation': None},
    'AudioShake':           {'dialogue_clarity': None, 'music_preservation': None},
    'EL-AudioIsolation':    {'dialogue_clarity': None, 'music_preservation': None},
}
for name, s in subjective.items():
    print(f'{name:25s}  clarity={s["dialogue_clarity"]}  music={s["music_preservation"]}')

## Ensemble: Spectrogram Averaging

Averaging the magnitude spectrograms of BS-RoFormer + MelBand-RoFormer before inverse STFT reduces artefacts that are model-specific. This is a well-known trick in the audio separation community.


In [ ]:
import soundfile as sf

# ── Weighted ensemble of vocal + instrumental stems ───────────────────────────
# Configurable weights — must sum to 1.0 across models that actually ran.
# Models that failed are skipped automatically; weights are re-normalised.
ENSEMBLE_WEIGHTS = {
    'BS-RoFormer-official': 0.40,
    'MelBand-RoFormer':     0.40,
    'HTDemucs-FT':          0.20,
}

# ── Step 1: build HTDemucs instrumental (bass + drums + other) ───────────────
htd = results.get('HTDemucs-FT', {})
htd_dir = os.path.join(RESULTS_DIR, 'HTDemucs-FT')
if htd.get('vocals') and not (htd.get('instrumental') and os.path.exists(htd.get('instrumental', ''))):
    stems_to_sum = []
    for fn in os.listdir(htd_dir):
        fl = fn.lower()
        if any(k in fl for k in ('bass', 'drums', 'other')) and fn.endswith('.wav'):
            y_s, sr_s = sf.read(os.path.join(htd_dir, fn), always_2d=True); y_s = y_s.T
            stems_to_sum.append(y_s)
    if stems_to_sum:
        max_l = max(s.shape[-1] for s in stems_to_sum)
        padded = [np.pad(s, ((0,0),(0,max_l-s.shape[-1]))) if s.ndim>1
                  else np.pad(s,(0,max_l-len(s))) for s in stems_to_sum]
        y_instr = np.sum(padded, axis=0)
        instr_path = os.path.join(htd_dir, 'instrumental.wav')
        sf.write(instr_path, y_instr.T if y_instr.ndim>1 else y_instr, sr_s)
        results['HTDemucs-FT']['instrumental'] = instr_path
        print(f'HTDemucs-FT instrumental reconstructed from {len(stems_to_sum)} stems -> {instr_path}')

# ── Step 2: weighted ensemble for vocals and instrumental ────────────────────
def weighted_ensemble(track_key):
    arrays, weights = [], []
    for name, w in ENSEMBLE_WEIGHTS.items():
        r = results.get(name, {})
        path = r.get(track_key)
        if path and os.path.exists(path):
            y, sr = sf.read(path, always_2d=True); y = y.T
            arrays.append(y)
            weights.append(w)
            print(f'  {name} ({track_key}): shape {y.shape}  weight={w}')
        else:
            print(f'  {name} ({track_key}): skipped (no output)')
    if len(arrays) < 2:
        print(f'  Need >= 2 models for ensemble {track_key}, skipping.')
        return None, None
    # Renormalise weights so they sum to 1
    total_w = sum(weights)
    weights = [w / total_w for w in weights]
    max_len = max(a.shape[-1] for a in arrays)
    padded  = [np.pad(a, ((0,0),(0,max_len-a.shape[-1]))) if a.ndim>1
               else np.pad(a,(0,max_len-len(a))) for a in arrays]
    result  = sum(w * a for w, a in zip(weights, padded))
    return result, sr

print('=== Ensemble Vocals ===')
y_ens_v, sr_v = weighted_ensemble('vocals')
print()
print('=== Ensemble Instrumental ===')
y_ens_i, sr_i = weighted_ensemble('instrumental')

ens_dir = os.path.join(STEMS_DIR, 'model_outputs', 'Ensemble')
os.makedirs(ens_dir, exist_ok=True)
ens_vocals_out = os.path.join(ens_dir, 'vocals.wav')
ens_instr_out  = os.path.join(ens_dir, 'instrumental.wav')

if y_ens_v is not None:
    sf.write(ens_vocals_out, y_ens_v.T if y_ens_v.ndim>1 else y_ens_v, sr_v)
    print(f'Ensemble vocals saved -> {ens_vocals_out}')
    print(f"Ensemble vocals: {ens_vocals_out}")

if y_ens_i is not None:
    sf.write(ens_instr_out, y_ens_i.T if y_ens_i.ndim>1 else y_ens_i, sr_i)
    print(f'Ensemble instrumental saved -> {ens_instr_out}')

results['Ensemble'] = {
    'vocals':       ens_vocals_out if y_ens_v is not None else None,
    'instrumental': ens_instr_out  if y_ens_i is not None else None,
    'rtf':          None,
    'error':        None if y_ens_v is not None else 'Not enough models succeeded',
}
print(f'Weights used (renormalised): { {n: round(ENSEMBLE_WEIGHTS[n]/sum(ENSEMBLE_WEIGHTS[v] for v in ENSEMBLE_WEIGHTS if results.get(v,{}).get("vocals")), 3) for n in ENSEMBLE_WEIGHTS if results.get(n,{}).get("vocals")} }')


In [ ]:
# Silero VAD on instrumentals — detects speech leaking into the music bed
# A clean instrumental should be < 5% speech frames.
# Install once: pip install silero-vad
import soundfile as sf
import numpy as np

_VAD_TRACKS = {
    "BS-RoFormer-official": os.path.join(STEMS_DIR, "model_outputs", "BS-RoFormer-official", "instrumental.wav"),
    "MelBand-RoFormer":     os.path.join(STEMS_DIR, "model_outputs", "MelBand-RoFormer",     "instrumental.wav"),
    "HTDemucs-FT":          os.path.join(STEMS_DIR, "model_outputs", "HTDemucs-FT",          "instrumental.wav"),
    "Ensemble":             os.path.join(STEMS_DIR, "model_outputs", "Ensemble",             "instrumental.wav"),
}

try:
    import torch
    print("Loading Silero VAD...")
    vad_model, utils = torch.hub.load(
        repo_or_dir="snakers4/silero-vad",
        model="silero_vad",
        force_reload=False, onnx=False, verbose=False,
    )
    (get_speech_timestamps, _, _, _, _) = utils
    vad_model.eval()
    print("Ready.\n")

    for name, path in _VAD_TRACKS.items():
        if not os.path.exists(path):
            print(f"[{name}] Skipped — file not found")
            continue
        y_raw, sr_raw = sf.read(path, always_2d=True)
        y_mono = y_raw.mean(axis=1).astype("float32")
        if sr_raw != 16000:
            import librosa as _lr
            y_mono = _lr.resample(y_mono, orig_sr=sr_raw, target_sr=16000)
        y_t = torch.from_numpy(y_mono)
        with torch.no_grad():
            speech_ts = get_speech_timestamps(y_t, vad_model, sampling_rate=16000,
                                              threshold=0.5, min_speech_duration_ms=200)
        speech_pct = sum(t["end"] - t["start"] for t in speech_ts) / len(y_mono) * 100
        rms = float(np.sqrt(np.mean(y_raw ** 2)))
        verdict = "CLEAN" if speech_pct < 5 else ("MODERATE" if speech_pct < 15 else "HEAVY BLEED")
        print(f"{name:25s}  speech={speech_pct:5.1f}%  rms={rms:.4f}  {verdict}")

    print("\n< 5% = clean   5-15% = moderate   >15% = heavy bleed")
except ImportError:
    print("silero-vad not installed. Run in terminal: " + "pip install silero-vad")
except Exception as e:
    print(f"Silero VAD failed: {e}")


In [ ]:
import shutil, json as _json

# ── Pick your winner ──────────────────────────────────────────────────────────
WINNER = 'BS-RoFormer-official'   # SOTA default; revisit after multi-episode evaluation

CANDIDATE_VOCALS = {
    'Ensemble':              os.path.join(STEMS_DIR, 'model_outputs', 'Ensemble', 'vocals.wav'),
    'BS-RoFormer-official':  os.path.join(STEMS_DIR, 'model_outputs', 'BS-RoFormer-official', 'vocals.wav'),
    'MelBand-RoFormer':      os.path.join(STEMS_DIR, 'model_outputs', 'MelBand-RoFormer', 'vocals.wav'),
    'HTDemucs-FT':           os.path.join(STEMS_DIR, 'model_outputs', 'HTDemucs-FT', 'vocals.wav'),
    'MDX-Net':               os.path.join(STEMS_DIR, 'model_outputs', 'MDX-Net', 'vocals.wav'),
}
CANDIDATE_INSTR = {
    'Ensemble':              os.path.join(STEMS_DIR, 'model_outputs', 'Ensemble', 'instrumental.wav'),
    'BS-RoFormer-official':  os.path.join(STEMS_DIR, 'model_outputs', 'BS-RoFormer-official', 'instrumental.wav'),
    'MelBand-RoFormer':      os.path.join(STEMS_DIR, 'model_outputs', 'MelBand-RoFormer', 'instrumental.wav'),
    'HTDemucs-FT':           os.path.join(STEMS_DIR, 'model_outputs', 'HTDemucs-FT', 'instrumental.wav'),
    'MDX-Net':               os.path.join(STEMS_DIR, 'model_outputs', 'MDX-Net', 'instrumental.wav'),
}

dst_v = os.path.join(STEMS_DIR, 'vocals.wav')
dst_i = os.path.join(STEMS_DIR, 'instrumental.wav')

vocals_src = CANDIDATE_VOCALS.get(WINNER)
instr_src  = CANDIDATE_INSTR.get(WINNER)

if vocals_src and os.path.exists(vocals_src):
    shutil.copy2(vocals_src, dst_v)
    print(f'Winner: {WINNER}')
    print(f'  vocals       -> {dst_v}')
else:
    print(f'ERROR: no vocals found for {WINNER} at {vocals_src}')

if instr_src and os.path.exists(instr_src):
    shutil.copy2(instr_src, dst_i)
    print(f'  instrumental -> {dst_i}')
else:
    fallback = CANDIDATE_INSTR['BS-RoFormer-official']
    if os.path.exists(fallback):
        shutil.copy2(fallback, dst_i)
        print(f'  instrumental -> {dst_i}  (fallback: BS-RoFormer-official)')
    else:
        print('ERROR: no instrumental available — assembly will fail')

meta = {'winner': WINNER, 'vocals': dst_v, 'instrumental': dst_i}
with open(os.path.join(STEMS_DIR, 'meta.json'), 'w') as f:
    _json.dump(meta, f, indent=2)
print('meta.json saved. Ready for notebook 03.')


In [ ]:
import soundfile as sf, numpy as np
# RMS comparison of each model's instrumental (uses STEMS_DIR from cell-setup)
_models = {
    "HTDemucs-FT":          os.path.join(STEMS_DIR, "model_outputs", "HTDemucs-FT",          "instrumental.wav"),
    "BS-RoFormer-official": os.path.join(STEMS_DIR, "model_outputs", "BS-RoFormer-official", "instrumental.wav"),
    "MelBand-RoFormer":     os.path.join(STEMS_DIR, "model_outputs", "MelBand-RoFormer",     "instrumental.wav"),
    "Ensemble":             os.path.join(STEMS_DIR, "model_outputs", "Ensemble",             "instrumental.wav"),
}
for _name, _path in _models.items():
    if os.path.exists(_path):
        _y, _ = sf.read(_path, always_2d=True)
        print(f"{_name:25s}  instr RMS: {float(np.sqrt(np.mean(_y**2))):.4f}")
    else:
        print(f"{_name:25s}  not found: {_path}")
